# HealthRAG — Etapa 1: entorno y catálogo del corpus

La estructura del proyecto y los módulos ya forman parte del repositorio; este notebook solo **usa** esos módulos (la lógica no vive en el notebook).

1. Copiar los PDF a `data/raw/`.
2. Completar `data/metadata_template.csv` (una fila por PDF; `file_name` debe coincidir exactamente). Usar los `doc_id` que aparecen en `data/evaluation/queries.csv` (p. ej. `GPCN_HTA`, `GPCN_IAM`, `GPCN_prev_cardiovascular`, `Guia_OMS_prev_ECV`, `Directriz_OMS_HTA`).
3. Ejecutar las celdas.

In [1]:
# Ejecutar una sola vez por entorno (o usar: pip install -r requirements.txt)
# %pip install -q -r ../requirements.txt

In [2]:
import os, sys
from pathlib import Path

# Raíz del proyecto: funciona abriendo el notebook desde notebooks/ o desde la raíz.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from app.config import load_config, resolve_path
from app.utils.logging_config import configure_logging

cfg = load_config()
logger = configure_logging(cfg)
print("Proyecto:", ROOT)

Proyecto: HealthRAG


In [3]:
import pandas as pd
raw_dir = resolve_path(cfg['corpus']['raw_dir'])
pdfs = sorted(raw_dir.rglob('*.pdf'))
print(f'PDF encontrados: {len(pdfs)}')
for p in pdfs: print('-', p.name)
display(pd.read_csv(resolve_path(cfg['corpus']['metadata_path'])))

PDF encontrados: 5
- Directriz-OMS-tto-farmaco-HTA.pdf
- GPCN-HTA-version_breve.pdf
- GPCN-IAM.pdf
- GPCN-prev-cardiovascular.pdf
- Guia-OMS-prev-enf-cardiovascular.pdf


,file_name,doc_id,title,organization,publication_year,version,section,page,source_url,notes
0,GPCN-HTA-version_breve.pdf,GPCN_HTA,Guía de Práctica Clínica Nacional de Prevenció...,Ministerio de Salud de la Nación (Argentina),2026,Versión breve 2026 de la guía actualizada 2024,NaN,NaN,https://www.argentina.gob.ar/sites/default/fil...,Año y versión tomados de la portada y la pág. ...
1,GPCN-IAM.pdf,GPCN_IAM,Guía de Práctica Clínica Nacional para la Repe...,Ministerio de Salud de la Nación (Argentina),2023,Versión aprobada por Resolución MS 256/2023,NaN,NaN,https://argentina.gob.ar/sites/default/files/b...,Aprobada por Resolución MS 256/2023 (BO 23/02/...
2,GPCN-prev-cardiovascular.pdf,GPCN_prev_cardiovascular,Guía de Práctica Clínica Nacional sobre Preven...,Ministerio de Salud de la Nación (Argentina),2021,Versión extensa,NaN,NaN,https://bancos.salud.gob.ar/recurso/guia-de-pr...,Año no explícito en el PDF (revisado página po...
3,Guia-OMS-prev-enf-cardiovascular.pdf,Guia_OMS_prev_ECV,Prevención de las enfermedades cardiovasculare...,Organización Mundial de la Salud,2008,Edición en español con tablas OMS/ISH para las...,NaN,NaN,https://iris.who.int/handle/10665/43847,ISBN 978 92 4 354728 2 (pág. 3). Guía de 2008:...
4,Directriz-OMS-tto-farmaco-HTA.pdf,Directriz_OMS_HTA,Directriz para el tratamiento farmacológico de...,Organización Mundial de la Salud,2022,Traducción al español de la edición en inglés ...,NaN,NaN,https://iris.who.int/handle/10665/364491,ISBN 978-92-4-006110-1 (pág. 4). Licencia CC B...


In [4]:
from app.ingestion.pdf_catalog import build_corpus_catalog

catalog = build_corpus_catalog(
    raw_dir=raw_dir,
    output_csv=resolve_path(cfg['corpus']['catalog_path']),
    min_text_chars_per_page=cfg['corpus']['min_text_chars_per_page'],
    metadata_csv=resolve_path(cfg['corpus']['metadata_path']),
)
display(catalog)

2026-11-15 12:21:18,576 | INFO | healthrag.ingestion.catalog | PDF inspeccionados: 5


2026-11-15 12:21:18,639 | INFO | healthrag.ingestion.catalog | Catálogo guardado en HealthRAG\data\processed\corpus_catalog.csv


,doc_id,title,organization,publication_year,version,section,page,source_url,file_name,file_path,sha256,page_count,pages_with_text,total_text_chars,is_probably_scanned,status,notes
0,Directriz_OMS_HTA,Directriz para el tratamiento farmacológico de...,Organización Mundial de la Salud,2022,Traducción al español de la edición en inglés ...,NaN,NaN,https://iris.who.int/handle/10665/364491,Directriz-OMS-tto-farmaco-HTA.pdf,HealthRA...,9ed14e41f8b6ef89975960c0a23538b3c0bb4469447c8a...,64,61,183337,False,ok,ISBN 978-92-4-006110-1 (pág. 4). Licencia CC B...
1,GPCN_HTA,Guía de Práctica Clínica Nacional de Prevenció...,Ministerio de Salud de la Nación (Argentina),2026,Versión breve 2026 de la guía actualizada 2024,NaN,NaN,https://www.argentina.gob.ar/sites/default/fil...,GPCN-HTA-version_breve.pdf,HealthRA...,f21863e081a4331a822eb348b1b792644e03edd9146e87...,40,40,70274,False,ok,Año y versión tomados de la portada y la pág. ...
2,GPCN_IAM,Guía de Práctica Clínica Nacional para la Repe...,Ministerio de Salud de la Nación (Argentina),2023,Versión aprobada por Resolución MS 256/2023,NaN,NaN,https://argentina.gob.ar/sites/default/files/b...,GPCN-IAM.pdf,HealthRA...,d27851a46a3813c1b7d0adf47f0bb0dd60dd7952c8264f...,80,78,219225,False,ok,Aprobada por Resolución MS 256/2023 (BO 23/02/...
3,GPCN_prev_cardiovascular,Guía de Práctica Clínica Nacional sobre Preven...,Ministerio de Salud de la Nación (Argentina),2021,Versión extensa,NaN,NaN,https://bancos.salud.gob.ar/recurso/guia-de-pr...,GPCN-prev-cardiovascular.pdf,HealthRA...,6088ec39b2ea5f51519111b2e42d7a5a4372fec30b4299...,176,150,359691,False,ok,Año no explícito en el PDF (revisado página po...
4,Guia_OMS_prev_ECV,Prevención de las enfermedades cardiovasculare...,Organización Mundial de la Salud,2008,Edición en español con tablas OMS/ISH para las...,NaN,NaN,https://iris.who.int/handle/10665/43847,Guia-OMS-prev-enf-cardiovascular.pdf,HealthRA...,347c9837ba0a56033a82974900cb32f17ed15fbf5dbf27...,38,36,52628,False,ok,ISBN 978 92 4 354728 2 (pág. 3). Guía de 2008:...


In [5]:
if catalog.empty:
    print('Corpus vacío: copiar PDF a data/raw/')
else:
    print('Documentos:', len(catalog))
    display(catalog['status'].value_counts().to_frame('cantidad'))
    print('Posibles escaneados:', int(catalog['is_probably_scanned'].astype(bool).sum()))
    print('Sin metadatos (doc_id):', int((catalog['status'] == 'missing_metadata').sum()))

Documentos: 5


,cantidad
status,
ok,5


Posibles escaneados: 0
Sin metadatos (doc_id): 0


**Criterio de cierre:** todos los PDF en estado `ok`; los `review_required` (escaneados) requieren OCR y los `duplicate_*` / `missing_metadata` deben resolverse antes de la Etapa 2.